In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W
from pyspark.sql import SparkSession

In [2]:
df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
df.show(20, truncate=False)


+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|order_id  |sku_id|item_name            |order_date|quantity|unit_price|discount_pct|amount|warehouse_id|customer_id|delivery_partner_id|order_status|dispatch_date|delivered_date|promised_date|shipping_cost|sku_rating|
+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|ORD0006769|SKU003|Sunflower Oil 1L     |2025-03-01|4       |215.98    |0.02        |846.64|WH02        |C00003     |DP02               |DELIVERED   |2025-03-03   |2025-03-05    |2025-03-07   |114.51       |4         |
|ORD0006769|SKU002|Whole Wheat Flour 5kg|2025-03-01|1       |306.43    |0.02        |300.3 |WH01        |C00003     |DP06   

In [3]:
df = df.withColumn('month',F.date_format(F.col('order_date'),'MMM'))
df.show(20, truncate=False)

+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+-----+
|order_id  |sku_id|item_name            |order_date|quantity|unit_price|discount_pct|amount|warehouse_id|customer_id|delivery_partner_id|order_status|dispatch_date|delivered_date|promised_date|shipping_cost|sku_rating|month|
+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+-----+
|ORD0006769|SKU003|Sunflower Oil 1L     |2025-03-01|4       |215.98    |0.02        |846.64|WH02        |C00003     |DP02               |DELIVERED   |2025-03-03   |2025-03-05    |2025-03-07   |114.51       |4         |Mar  |
|ORD0006769|SKU002|Whole Wheat Flour 5kg|2025-03-01|1       |306.43    |0.02        |300.3 |WH01    

In [4]:
df_groupby = (
    df.groupBy("month", "sku_id", "item_name")
    .agg(F.sum("quantity").alias("total_order_per_month"))
    .select("sku_id", "item_name", "month", "total_order_per_month")
    .orderBy(F.col("total_order_per_month").desc())
)
df_groupby.show(20, truncate=False)

+------+---------------------+-----+---------------------+
|sku_id|item_name            |month|total_order_per_month|
+------+---------------------+-----+---------------------+
|SKU012|Eggs Tray 30         |Jan  |1425                 |
|SKU016|Instant Noodles      |Jun  |1409                 |
|SKU012|Eggs Tray 30         |Jun  |1389                 |
|SKU016|Instant Noodles      |Apr  |1318                 |
|SKU012|Eggs Tray 30         |Apr  |1277                 |
|SKU012|Eggs Tray 30         |Mar  |1270                 |
|SKU012|Eggs Tray 30         |Feb  |1225                 |
|SKU016|Instant Noodles      |Jan  |1223                 |
|SKU012|Eggs Tray 30         |May  |1203                 |
|SKU016|Instant Noodles      |Feb  |1203                 |
|SKU016|Instant Noodles      |Mar  |1202                 |
|SKU011|Bread Loaf           |Jan  |1169                 |
|SKU016|Instant Noodles      |May  |1126                 |
|SKU002|Whole Wheat Flour 5kg|Jun  |1088                

In [5]:
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Load table and extract the 3-letter month name from order_date
df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
df = df.withColumn("month", F.date_format(F.col("order_date"), "MMM"))

# 2. Aggregate monthly orders per SKU
df_groupby = (
    df.groupBy("month", "sku_id")
    .agg(F.sum("quantity").alias("total_order_per_month"))
    .select("sku_id", "month", "total_order_per_month")
)

# 3. Convert to Pandas & prepare Month 7 (Jul)
pdf = df_groupby.toPandas()

month_map = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6}
pdf["month_num"] = pdf["month"].map(month_map)
pdf = pdf.sort_values(["sku_id", "month_num"]).reset_index(drop=True)

skus = pdf[["sku_id"]].drop_duplicates()
month_7 = skus.copy()
month_7["month"] = "Jul"
month_7["month_num"] = 7
month_7["total_order_per_month"] = float("nan")

full_df = pd.concat([pdf, month_7], ignore_index=True)
full_df = full_df.sort_values(["sku_id", "month_num"]).reset_index(drop=True)
full_df["total_order_per_month"] = full_df["total_order_per_month"].astype(float)

# 4. Feature Engineering
full_df["sku_code"] = full_df["sku_id"].astype("category").cat.codes
full_df["lag_1"] = full_df.groupby("sku_id")["total_order_per_month"].shift(1)
full_df["lag_2"] = full_df.groupby("sku_id")["total_order_per_month"].shift(2)
full_df["rolling_mean_2"] = (full_df["lag_1"] + full_df["lag_2"]) / 2
full_df["diff_1_2"] = full_df["lag_1"] - full_df["lag_2"]

features = ["sku_code", "month_num", "lag_1", "lag_2", "rolling_mean_2", "diff_1_2"]

train_df = full_df[full_df["month_num"] <= 6].dropna(subset=["lag_2"])
pred_df = full_df[full_df["month_num"] == 7].copy()

# 5. Train Model & Forecast Month 7
model = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
model.fit(train_df[features], train_df["total_order_per_month"])

pred_df["forecast_month_7"] = model.predict(pred_df[features]).round(0).astype(int)

# 6. Display Results
result = pred_df[["sku_id", "lag_2", "lag_1", "forecast_month_7"]].rename(
    columns={"lag_2": "May_actual", "lag_1": "Jun_actual"}
)

print(result.sort_values("forecast_month_7", ascending=False).to_string(index=False))


sku_id  May_actual  Jun_actual  forecast_month_7
SKU012      1215.0      1401.0              1125
SKU016      1147.0      1423.0              1125
SKU002      1033.0      1097.0              1081
SKU011       915.0      1046.0              1034
SKU001       762.0       906.0               896
SKU015       776.0       815.0               803
SKU013       840.0       820.0               788
SKU004       782.0       889.0               754
SKU008       792.0       889.0               754
SKU029       676.0       726.0               705
SKU003       624.0       732.0               672
SKU021       530.0       578.0               665
SKU019       411.0       537.0               571
SKU006       374.0       414.0               441
SKU014       345.0       399.0               424
SKU005       337.0       370.0               400
SKU010       281.0       391.0               397
SKU022       341.0       357.0               391
SKU020       314.0       315.0               346
SKU030       307.0  

In [6]:
result.write.mode('overwrite')\
            .saveAsTable("dbt_adv_pr.gold.month_7_forecast")

AttributeError: 'DataFrame' object has no attribute 'write'

In [ ]:
df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

monthly_df = (
    df.withColumn("month_num",F.month("order_date"))
    .withColumn('month',F.date_format("order_date",'MMM'))
    .groupBy('sku_id','month_num','month')
    .agg(F.sum('quantity').alias('monthly_qty'))
)

sku_window = W.partitionBy('sku_id')

stats_df = (
    monthly_df
    .withColumn('avg_monthly_qty',F.round(F.avg('monthly_qty').over(sku_window),2))
    .withColumn('std_monthly_qty',F.round(F.stddev('monthly_qty').over(sku_window),2))
    .withColumn("max_monthly_qty",F.max('monthly_qty').over(sku_window))
    .withColumn('min_monthly_qty',F.min('monthly_qty').over(sku_window))
    .withColumn("seasonality_index",F.round(F.col('monthly_qty') / F.col('avg_monthly_qty'),2))
)

sku_seasonality = (
    stats_df.groupBy('sku_id','avg_monthly_qty','std_monthly_qty','max_monthly_qty','min_monthly_qty')
    .agg(
        F.max(F.when(F.col('monthly_qty') == F.col('max_monthly_qty'),F.col('month'))).alias('peak_month'),
        F.min(F.when(F.col('monthly_qty') == F.col('min_monthly_qty'),F.col('month'))).alias('low_month'),
        F.count(F.when(F.col('seasonality_index') >= 1.20, True)).alias('peak_months_count')
    )
    .withColumn('cv',F.round(F.col('std_monthly_qty') / F.col('avg_monthly_qty'),2))
    .withColumn('peak_to_avg_ratio',F.round(F.col('max_monthly_qty') / F.col('avg_monthly_qty'),2))
    .withColumn(
        "demand_pattern",
        F.when(F.col('cv') < 0.15, "Steady(Non-Seaso)")
         .when((F.col('cv')>=0.15) & (F.col('peak_months_count') >= 2), "Seasonal / Cyclical")
         .otherwise("Single-Month Spike / Erratic")
    )
    .orderBy(F.desc('cv'))
)

sku_seasonality.show(20, truncate=False)

+------+---------------+---------------+---------------+---------------+----------+---------+-----------------+----+-----------------+----------------------------+
|sku_id|avg_monthly_qty|std_monthly_qty|max_monthly_qty|min_monthly_qty|peak_month|low_month|peak_months_count|cv  |peak_to_avg_ratio|demand_pattern              |
+------+---------------+---------------+---------------+---------------+----------+---------+-----------------+----+-----------------+----------------------------+
|SKU037|37.33          |48.21          |135            |12             |Mar       |Feb      |1                |1.29|3.62             |Single-Month Spike / Erratic|
|SKU038|47.83          |58.03          |166            |20             |Mar       |Apr      |1                |1.21|3.47             |Single-Month Spike / Erratic|
|SKU036|37.33          |32.28          |101            |19             |Mar       |Feb      |1                |0.86|2.71             |Single-Month Spike / Erratic|
|SKU035|38.0    

In [ ]:
sku_seasonality.write.mode('overwrite')\
                     .saveAsTable('dbt_adv_pr.gold.seasonal_test')

In [ ]:
sku_cv = sku_seasonality.drop('demand_pattern')
sku_cv.show(20, truncate = False)

+------+---------------+---------------+---------------+---------------+----------+---------+-----------------+----+-----------------+
|sku_id|avg_monthly_qty|std_monthly_qty|max_monthly_qty|min_monthly_qty|peak_month|low_month|peak_months_count|cv  |peak_to_avg_ratio|
+------+---------------+---------------+---------------+---------------+----------+---------+-----------------+----+-----------------+
|SKU037|37.33          |48.21          |135            |12             |Mar       |Feb      |1                |1.29|3.62             |
|SKU038|47.83          |58.03          |166            |20             |Mar       |Apr      |1                |1.21|3.47             |
|SKU036|37.33          |32.28          |101            |19             |Mar       |Feb      |1                |0.86|2.71             |
|SKU035|38.0           |28.51          |66             |9              |Feb       |Mar      |1                |0.75|1.74             |
|SKU040|39.67          |29.4           |64             

In [ ]:
sku_cv.write.mode('overwrite')\
            .saveAsTable("dbt_adv_pr.gold.sku_cv")